In [5]:
import tensorflow as tf 
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 하이퍼파라미터 설정
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATA_DIR = '../../data/dataset_imagenet'

# 이미지 데이터 생성
# 학습 데이터
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=1000,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 20 files belonging to 2 classes.
Using 16 files for training.


In [18]:
# 검증데이터
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=1000,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = val_ds.class_names

Found 20 files belonging to 2 classes.
Using 4 files for validation.


In [6]:
# 성능 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(20).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [8]:
# 모델 설계
# 전이학습 모델 가져옴
base_model = MobileNetV2(input_shape=(224, 224,  3),
                         include_top=False, weights='imagenet')
# 데이터 증강 => 이미지를 돌리고 확대하고 움직여서 여러 이미지를 추가하는 작업
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])

# 새 모델 설계
model = models.Sequential([
    layers.Input((224,224,3)),
    data_augmentation,
    # 정규화를 0~1이 아닌 -1~-1로 해야함. 왜? imagenet에서 그렇게 했기 때문에 맞춰야 함
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(2, activation='softmax')
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [11]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
              

In [10]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 32s 32s/step - accuracy: 0.4375 - loss: 0.7725 - val_accuracy: 1.0000 - val_loss: 0.0556
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0939 - val_accuracy: 1.0000 - val_loss: 0.0136
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0093 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0173 - val_accuracy: 1.0000 - val_loss: 5.6505e-04
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0065 - val_accuracy: 1.0000 - val_loss: 1.9989e-04
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 3.8706e-04 - val_accuracy: 1.0000 - val_loss: 9.3681e-05
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 4.7216e-04 - val_accuracy: 1.0000 - val_loss: 5.2536e-05
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 1.0000 - val_los

In [ ]:
import requests
import numpy as np
from io import BytesIO
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import mobilenet_v2

def predict_with_url(url):
    resp = requests.get(url)
    img = image.load_img(BytesIO(resp.content), target_size=(224, 224))
    X = image.img_to_array(img)
    X = np.expand_dims(X, axis=0)
    # MobileNet v2 전용 전처리
    # X = mobilenet_v2.preprocess_input(X)

    # 예측
    p = model.predict(X)
    print(p, class_names)
    i = np.argmax(p)
    print("="*30)
    print(f'{class_names[i]}: {p[0][i]*100:.2f}%')
    print("="*30)

In [54]:
# 떡볶이
predict_with_url('https://mblogthumb-phinf.pstatic.net/MjAyNDA0MjZfMTg3/MDAxNzE0MTI4NzQwNTM1.nFy_4HtjUUSVsD0CBFcKcR1FaUsrsAF-1AcJqKDx-gog.5hV0YZ5uDLR_xOmSZUOsLFvqE3GOJ4yi7LigslRYlW4g.JPEG/SE-10362da0-8490-409d-a6b2-06550fad5ee2.jpg?type=w800')
# 김밥
predict_with_url('https://wooltariusa.com/cdn/shop/files/12t_847f7a04-c9d3-4311-aa84-28eaf041b435.jpg?v=1713720002&width=1000')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
[[0.00473851 0.9952615 ]] ['kimbap', 'tteokbokki']
tteokbokki: 99.53%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
[[0.00334601 0.9966539 ]] ['kimbap', 'tteokbokki']
tteokbokki: 99.67%
